# EvalConvoLearn — Evaluation Results Analysis

Loads the output of `eedi_fitted_learner_evals.py` and provides:
1. **Results comparison table** across learner types, models, and few-shot counts
2. **Adjustable metric weights** — re-score without re-running the eval
3. **Conversation browser** — inspect specific runs qualitatively
4. **Random sampling for human validation** — pull generated conversations with LLM-as-judge classifications (talk moves, error types) for manual grading

In [ ]:
import json
import random
from pathlib import Path

import pandas as pd
from IPython.display import display

## 1 · Configuration

Point `EVAL_OUTPUT_DIR` at the root directory that contains your evaluation run sub-folders (produced by `eedi_fitted_learner_evals.py`).

In [ ]:
# Path to the folder in dataset_fitted_evals/
# eval_dir = "binary_skills_learner/binary_skills__gpt-4_1-mini__fs3__20260602_07-52-53"

# EVAL_OUTPUT_DIR = Path("outputs/dataset_fitted_evals/") / eval_dir
EVAL_OUTPUT_DIR = Path("outputs/dataset_fitted_evals/")

## 2 · Load Summary Files

In [ ]:
def load_summaries(root: Path) -> list[dict]:
    summaries = []
    for path in sorted(root.rglob("dataset_fitted_conversational_summary.json")):
        with path.open(encoding="utf-8") as fh:
            data = json.load(fh)
        data["_summary_path"] = str(path)
        data["_details_dir"] = str(path.parent)
        summaries.append(data)
    return summaries


summaries = load_summaries(EVAL_OUTPUT_DIR)
print(f"Found {len(summaries)} summary file(s)")
for s in summaries:
    score = s.get("eval_convo_learn_score")
    score_str = f"{score:.4f}" if score is not None else "N/A"
    print(f"  {s.get('learner_label', 'unknown')}  ->  score={score_str}")

## 3 · Results Comparison Table

In [ ]:
def _parse_label(label: str) -> dict:
    # Expected format: {learner_type}__{model}__{fs_tag}__{run_id}
    # As defined in eedi_fitted_learner_evals.py

    # Can be renamed to specific evaluations and extract other information in the analysis phase
    parts = label.split("__")
    return {
        "learner_type": parts[0] if len(parts) > 0 else label,
        "model_learner": parts[1].replace("_", ".") if len(parts) > 1 else "",
        "model_evals": parts[2].replace("_", ".") if len(parts) > 2 else "",
        "few_shot": parts[3] if len(parts) > 3 else "",
        "run_id": "__".join(parts[4:]) if len(parts) > 4 else "",
    }


def _get_se(standard_errors: dict, key: str) -> float | None:
    entry = standard_errors.get(key)
    return entry.get("se") if isinstance(entry, dict) else None


def build_results_df(summaries: list[dict]) -> pd.DataFrame:
    rows = []
    for s in summaries:
        label = s.get("learner_label", "")
        parsed = _parse_label(label)
        agg = s.get("aggregate_scores", {})
        ecl = agg.get("eval_convo_learn", {})
        counts = s.get("counts", {})
        se = s.get("standard_errors", {})
        n_runs = se.get("n_runs", 1)
        rows.append(
            {
                "learner_type": parsed["learner_type"],
                "model_learner": parsed["model_learner"],
                "model_evals": parsed["model_evals"],
                "few_shot": parsed["few_shot"],
                "run_id": parsed["run_id"],
                "ecl_score": ecl.get("score"),
                "ecl_se": _get_se(se, "eval_convo_learn_score"),
                "lb_score": ecl.get("lb_score"),
                "lb_se": _get_se(se, "lb_score"),
                "conv_score": ecl.get("conv_score"),
                "conv_se": _get_se(se, "conv_score"),
                "n_runs": n_runs,
                "alpha": ecl.get("alpha"),
                "solution_found_rate": s.get("solution_found", {}).get("overall_rate"),
                "n_simulated": counts.get("successful_simulated_records"),
                "n_real": counts.get("selected_skill_real_records"),
                "_summary_path": s["_summary_path"],
            }
        )
    return pd.DataFrame(rows)


results_df = build_results_df(summaries)
display(
    results_df.sort_values(["learner_type", "model_learner", "model_evals", "few_shot"])
    .drop(columns=["_summary_path"])
    .style.format(
        {
            "ecl_score": "{:.3f}",
            "ecl_se": "{:.3f}",
            "lb_score": "{:.3f}",
            "lb_se": "{:.3f}",
            "conv_score": "{:.3f}",
            "conv_se": "{:.3f}",
            "solution_found_rate": "{:.2f}",
            "alpha": "{:.2f}",
        },
        na_rep="N/A",
    )
)

In [ ]:
### Filter results for a specific model or few-shot setting
results_df = results_df[
    results_df["run_id"].str.contains("20260617")
    # & results_df["conv_se"].notna()
    # & results_df["model_learner"].str.contains("gpt-4.1-mini")
]
results_df

### 3b · Score Comparison with ±SE Error Bars

Shows ECL, LB, and CONV scores per run configuration. Error bars are ±1 SE across benchmark runs (only shown when `n_runs >= 2`).

In [ ]:
try:
    import matplotlib.pyplot as plt

    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("matplotlib not installed — skipping plot. Install with: pip install matplotlib")

if HAS_MATPLOTLIB and not results_df.empty:
    plot_df = results_df.sort_values(["learner_type", "model_learner", "model_evals", "few_shot"]).reset_index(
        drop=True
    )
    x_pos = range(len(plot_df))
    tick_labels = [f"{r.learner_type}\n{r.model_learner}\n{r.few_shot}" for r in plot_df.itertuples()]

    fig, ax = plt.subplots(figsize=(max(9, len(plot_df) * 0.85), 5))
    bar_width = 0.24
    offsets = {"ecl_score": -bar_width, "lb_score": 0.0, "conv_score": bar_width}
    colors = {"ecl_score": "steelblue", "lb_score": "salmon", "conv_score": "seagreen"}
    display_labels = {"ecl_score": "ECL score", "lb_score": "LB score", "conv_score": "CONV score"}

    for metric, offset in offsets.items():
        se_col = metric.replace("_score", "_se")
        scores = plot_df[metric].tolist()
        # Use 0 SE when not available (single run)
        ses = [v if v is not None else 0.0 for v in plot_df[se_col].tolist()]
        xs = [v + offset for v in x_pos]
        ax.bar(
            xs,
            scores,
            width=bar_width,
            color=colors[metric],
            label=display_labels[metric],
            alpha=0.85,
            yerr=ses,
            capsize=3,
            error_kw={"elinewidth": 1.2},
        )

    ax.set_xticks(list(x_pos))
    ax.set_xticklabels(tick_labels, fontsize=7, rotation=30, ha="right")
    ax.set_ylabel("Score (lower is better for LB/CONV, lower ECL is worse)")
    ax.set_title(
        "Score comparison across learner configurations\n(error bars = ±1 SE across runs; absent when n_runs < 2)"
    )
    ax.legend(loc="upper right")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

### 3c · Per-Metric Standard Errors

Select a run by label to inspect SE for each sub-metric.

In [ ]:
SE_SHOW_LABEL = summaries[-1]["learner_label"] if summaries else ""

_SE_METRIC_LABELS = {
    "eval_convo_learn_score": "ECL score",
    "lb_score": "LB score",
    "conv_score": "CONV score",
    "q_w1": "Questions W1",
    "w_w1_log": "AvgWords W1 (log)",
    "error_jsd": "Error-types JSD",
    "talk_jsd": "Talk-moves JSD",
}

se_target = next((s for s in summaries if s.get("learner_label") == SE_SHOW_LABEL), None)
if se_target is None:
    print(f"Label not found: {SE_SHOW_LABEL!r}")
else:
    se = se_target.get("standard_errors", {})
    n_runs = se.get("n_runs", 0)
    if n_runs < 2:
        print(f"SE not available: {se.get('note', f'only {n_runs} run(s)')}\n")
        per_run = se.get("per_run_scores", [])
        if per_run:
            display(
                pd.DataFrame(per_run).style.format(
                    "{:.4f}", subset=[c for c in pd.DataFrame(per_run).columns if c != "run_id"]
                )
            )
    else:
        se_rows = []
        for key, label in _SE_METRIC_LABELS.items():
            entry = se.get(key, {})
            if isinstance(entry, dict):
                se_rows.append(
                    {
                        "metric": label,
                        "mean": entry.get("mean"),
                        "se": entry.get("se"),
                        "std": entry.get("std"),
                        "values": str([round(v, 4) for v in entry.get("values", [])]),
                    }
                )
        print(f"Standard errors across {n_runs} runs  —  {SE_SHOW_LABEL}")
        display(pd.DataFrame(se_rows).style.format({"mean": "{:.4f}", "se": "{:.4f}", "std": "{:.4f}"}, na_rep="N/A"))

## 4 · Re-score with Adjustable Metric Weights

Change the variables below and re-run the cell to recompute scores without re-running the evaluation.

**Top-level weight** (`ALPHA`):  
`ecl_score = ALPHA * lb_score + (1 - ALPHA) * conv_score`

**Within the conversational score** (must sum to 1.0):

| Metric | Variable | Default |
|--------|----------|---------|
| Questions per interrogative turn (W1) | `W_Q_W1` | 0.15 |
| Avg words per learner turn — log-scale W1 | `W_W_W1_LOG` | 0.25 |
| Error type distribution JSD | `W_ERROR_JSD` | 0.30 |
| Talk move distribution JSD | `W_TALK_JSD` | 0.30 |

In [ ]:
### Filter summaries for a specific model or few-shot setting
# for example already filtered in results_df above, or on other properties

summaries = [s for s in summaries if s.get("_summary_path") in results_df._summary_path.values]

print(f"Filtered summaries count: {len(summaries)}")
for s in summaries:
    print(f"  {s.get('learner_label', 'unknown')}  ->  conv_se={s.get('conv_se')}")

In [ ]:
# ---- Adjust these weights ----
ALPHA = 0.5  # LB vs conversational
W_Q_W1 = 0.25  # questions per interrogative turn
W_W_W1_LOG = 0.25  # avg words per learner turn (log-scale W1)
W_ERROR_JSD = 0.25  # error type distribution JSD
W_TALK_JSD = 0.25  # talk move distribution JSD
# ------------------------------

assert abs(W_Q_W1 + W_W_W1_LOG + W_ERROR_JSD + W_TALK_JSD - 1.0) < 1e-9, "Conversational sub-weights must sum to 1.0"


def recompute_scores(summary: dict, alpha: float, weights: dict) -> dict:
    """Re-derive all scores from per-run data stored in standard_errors.

    Each metric is first computed per run (using the user-supplied weights),
    then averaged.  Because the mean is linear, conv_score == weighted sum of
    the four sub-metric means — and ecl_score == alpha*lb_mean + (1-alpha)*conv_mean.

    Falls back to the pooled aggregate_scores when only a single run is available
    (no SE can be reported in that case).
    """
    se = summary.get("standard_errors", {})
    per_run_scores = se.get("per_run_scores", [])

    def _stats(vals: list) -> dict:
        n = len(vals)
        mean = sum(vals) / n if n else 0.0
        if n < 2:
            return mean, None
        std = (sum((v - mean) ** 2 for v in vals) / (n - 1)) ** 0.5
        return mean, std / n**0.5

    if per_run_scores:
        run_q, run_w, run_e, run_t, run_conv, run_lb, run_ecl = [], [], [], [], [], [], []
        for run in per_run_scores:
            q = float(run.get("q_w1", 0.0))
            w = float(run.get("w_w1_log", 0.0))
            e = float(run.get("error_jsd", 0.0))
            t = float(run.get("talk_jsd", 0.0))
            lb = float(run.get("lb_score", 0.0))
            conv = weights["q_w1"] * q + weights["w_w1_log"] * w + weights["error_jsd"] * e + weights["talk_jsd"] * t
            ecl = alpha * lb + (1 - alpha) * conv
            run_q.append(q)
            run_w.append(w)
            run_e.append(e)
            run_t.append(t)
            run_conv.append(conv)
            run_lb.append(lb)
            run_ecl.append(ecl)

        q_mean, q_se = _stats(run_q)
        w_mean, w_se = _stats(run_w)
        e_mean, e_se = _stats(run_e)
        t_mean, t_se = _stats(run_t)
        conv_mean, conv_se = _stats(run_conv)
        lb_mean, lb_se = _stats(run_lb)
        ecl_mean, ecl_se = _stats(run_ecl)
        n_runs = len(per_run_scores)
    else:
        # Single-run fallback: derive from pooled aggregate_scores (no SE available)
        agg = summary.get("aggregate_scores", {})
        scenario_weights = agg.get("scenario_weights", {})
        lb_per_scen = agg.get("learning_behavior", {}).get("per_scenario", {})
        lb_wsum, lb_wtotal = 0.0, 0.0
        for scen, data in lb_per_scen.items():
            mae = data.get("macro_mae")
            if mae is not None:
                ww = scenario_weights.get(scen, data.get("weight", 0.0))
                lb_wsum += mae * ww
                lb_wtotal += ww
        lb_mean = lb_wsum / lb_wtotal if lb_wtotal > 0 else 0.0
        conv_per_scen = agg.get("conversational", {}).get("per_scenario", {})
        conv_wsum, conv_wtotal = 0.0, 0.0
        for scen, data in conv_per_scen.items():
            macro = (
                weights["q_w1"] * data.get("questions_per_interrogative_turn_w1", 0.0)
                + weights["w_w1_log"] * data.get("avg_words_per_learner_turn_w1_log", 0.0)
                + weights["error_jsd"] * data.get("error_types_jsd", 0.0)
                + weights["talk_jsd"] * data.get("talk_moves_jsd", 0.0)
            )
            ww = scenario_weights.get(scen, data.get("weight", 0.0))
            conv_wsum += macro * ww
            conv_wtotal += ww
        conv_mean = conv_wsum / conv_wtotal if conv_wtotal > 0 else 0.0
        ecl_mean = alpha * lb_mean + (1 - alpha) * conv_mean
        q_mean = w_mean = e_mean = t_mean = None
        q_se = w_se = e_se = t_se = conv_se = lb_se = ecl_se = None
        n_runs = 1

    return {
        "label": summary.get("learner_label", ""),
        "ecl_score": ecl_mean,
        "lb_score": lb_mean,
        "conv_score": conv_mean,
        "ecl_se": ecl_se,
        "lb_se": lb_se,
        "conv_se": conv_se,
        "q_w1": q_mean,
        "w_w1_log": w_mean,
        "error_jsd": e_mean,
        "talk_jsd": t_mean,
        "q_w1_se": q_se,
        "w_w1_log_se": w_se,
        "error_jsd_se": e_se,
        "talk_jsd_se": t_se,
        "n_runs": n_runs,
    }


weights = {"q_w1": W_Q_W1, "w_w1_log": W_W_W1_LOG, "error_jsd": W_ERROR_JSD, "talk_jsd": W_TALK_JSD}
rescored = [recompute_scores(s, ALPHA, weights) for s in summaries]
rescored_df = pd.DataFrame(
    [
        {
            **_parse_label(r["label"]),
            "ecl_score": r["ecl_score"],
            "lb_score": r["lb_score"],
            "conv_score": r["conv_score"],
            "ecl_se": r["ecl_se"],
            "lb_se": r["lb_se"],
            "conv_se": r["conv_se"],
            "q_w1": r["q_w1"],
            "w_w1_log": r["w_w1_log"],
            "error_jsd": r["error_jsd"],
            "talk_jsd": r["talk_jsd"],
            "q_w1_se": r["q_w1_se"],
            "w_w1_log_se": r["w_w1_log_se"],
            "error_jsd_se": r["error_jsd_se"],
            "talk_jsd_se": r["talk_jsd_se"],
            "n_runs": r["n_runs"],
        }
        for r in rescored
    ]
)

# Sanity-check: conv_score must equal the weighted sum of the four sub-metric means
_fails = [
    r["label"]
    for r in rescored
    if r["q_w1"] is not None
    and abs(
        r["conv_score"]
        - (
            weights["q_w1"] * r["q_w1"]
            + weights["w_w1_log"] * r["w_w1_log"]
            + weights["error_jsd"] * r["error_jsd"]
            + weights["talk_jsd"] * r["talk_jsd"]
        )
    )
    > 1e-9
]
if _fails:
    print(f"WARNING: conv_score != weighted sub-metric sum for: {_fails}")
else:
    print("✓ conv_score = weighted mean of sub-metrics for all runs")

print(f"Re-scored with alpha={ALPHA}, weights={weights}")
print("All scores are means of per-run values; SE computed across runs.")
display(
    rescored_df.sort_values(["learner_type", "model_learner", "model_evals", "few_shot"]).style.format(
        {
            "ecl_score": "{:.4f}",
            "lb_score": "{:.4f}",
            "conv_score": "{:.4f}",
            "ecl_se": "{:.4f}",
            "lb_se": "{:.4f}",
            "conv_se": "{:.4f}",
            "q_w1": "{:.4f}",
            "w_w1_log": "{:.4f}",
            "error_jsd": "{:.4f}",
            "talk_jsd": "{:.4f}",
            "q_w1_se": "{:.4f}",
            "w_w1_log_se": "{:.4f}",
            "error_jsd_se": "{:.4f}",
            "talk_jsd_se": "{:.4f}",
        },
        na_rep="N/A",
    )
)

In [ ]:
import math

# ── Shared helpers (also used by the Table 2 cell below) ─────────────────────

_LEARNER_NAMES = {
    "binary_skills": "Binary Skills",
    "conv_history": "Conv. History",
}


def _fmt(val: float | None, se: float | None = None, decimals: int = 3, bold: bool = False) -> str:
    """Format a metric value as a LaTeX cell string (val or val ± se)."""
    try:
        v = float(val)
        if math.isnan(v):
            return "---"
    except (TypeError, ValueError):
        return "---"
    d = f".{decimals}f"
    s = f"{v:{d}}"
    if se is not None:
        try:
            sv = float(se)
            if not math.isnan(sv):
                s += r" $\pm$ " + f"{sv:{d}}"
        except (TypeError, ValueError):
            pass
    return r"\textbf{" + s + "}" if bold else s


def _best_idx(df: "pd.DataFrame", col: str) -> object:
    """Row index with the lowest (best) numeric value in col."""
    return pd.to_numeric(df[col], errors="coerce").idxmin()


# ── Table 1 ───────────────────────────────────────────────────────────────────
# 2 selected learners: GPT-4.1-mini student, Sonnet 4.6 tutor, 3-shot.
# Stacked 5-column layout: sub-metric row + composite score row per learner.
# All values are means of per-run scores; SE is across the 3 runs.

t1 = (
    rescored_df[
        rescored_df["model_learner"].str.contains("gpt-4.1-mini", na=False)
        & rescored_df["model_evals"].str.contains("sonnet", na=False)
        & (rescored_df["few_shot"] == "fs3")
    ]
    .sort_values("learner_type")
    .reset_index(drop=True)
)

SUB_COLS = [
    ("talk_jsd", "talk_jsd_se"),
    ("error_jsd", "error_jsd_se"),
    ("q_w1", "q_w1_se"),
    ("w_w1_log", "w_w1_log_se"),
]
COMP_COLS = [
    ("conv_score", "conv_se"),
    ("lb_score", "lb_se"),
    ("ecl_score", "ecl_se"),
]

best = {c: _best_idx(t1, c) for c, _ in SUB_COLS + COMP_COLS}

lines = [
    r"\begin{tabular}{lcccc}",
    r"\toprule",
    r"\textbf{Setting} & \textbf{Talk Moves} & \textbf{Error Types} & \textbf{Questions} & \textbf{Turn Length} \\",
    r"                 & \textbf{CONV}       & \textbf{LB}         & \textbf{ECL}       &  \\",
    r"\midrule",
]

for i, row in t1.iterrows():
    name = _LEARNER_NAMES.get(row["learner_type"], row["learner_type"])
    sub = [_fmt(row[c], row[se], bold=(i == best[c])) for c, se in SUB_COLS]
    comp = [_fmt(row[c], row[se], bold=(i == best[c])) for c, se in COMP_COLS] + [""]
    lines.append(r"\multirow{2}{*}{" + name + r"} & " + " & ".join(sub) + r" \\")
    lines.append("  & " + " & ".join(comp) + r" \\")
    if i < len(t1) - 1:
        lines.append(r"\addlinespace")

lines += [r"\bottomrule", r"\end{tabular}"]

print("=== TABLE 1 ===\n")
print("\n".join(lines))

In [ ]:
# ── Table 2 (Annex) ───────────────────────────────────────────────────────────
# All configurations: both learner types × student models × few-shot setups.
# Bold = best (lowest) value in each score column across all rows.
# All values are means of per-run scores; SE is across the 3 runs.


def _model_display(raw: str) -> str:
    """Return a paper-friendly model name from the parsed label value."""
    name = raw[3:]  # strip "m1." / "m2." prefix
    if "gpt-4.1-mini" in name:
        return "GPT-4.1-mini"
    if "sonnet" in name.lower():
        return "Sonnet 4.6"
    return name


all_df = rescored_df.sort_values(["learner_type", "model_learner", "model_evals", "few_shot"]).reset_index(drop=True)

best2 = {c: _best_idx(all_df, c) for c in ["conv_score", "lb_score", "ecl_score"]}

lines2 = [
    r"\begin{tabular}{llllccc}",
    r"\toprule",
    (
        r"\textbf{Learner} & \textbf{Student} & \textbf{Tutor}"
        r" & \textbf{Few-shot} & \textbf{CONV} & \textbf{LB} & \textbf{ECL} \\"
    ),
    r"\midrule",
]

SCORE_COLS = [
    ("conv_score", "conv_se"),
    ("lb_score", "lb_se"),
    ("ecl_score", "ecl_se"),
]
prev_learner = None

for i, row in all_df.iterrows():
    learner = _LEARNER_NAMES.get(row["learner_type"], row["learner_type"])
    if prev_learner is not None and learner != prev_learner:
        lines2.append(r"\midrule")
    prev_learner = learner

    shot = "3-shot" if row["few_shot"] == "fs3" else "0-shot"
    cells = [
        learner,
        _model_display(row["model_learner"]),
        _model_display(row["model_evals"]),
        shot,
    ] + [_fmt(row[c], row[se], bold=(i == best2[c])) for c, se in SCORE_COLS]
    lines2.append(" & ".join(cells) + r" \\")

lines2 += [r"\bottomrule", r"\end{tabular}"]

print("=== TABLE 2 (annex) ===\n")
print("\n".join(lines2))

## 5 · Per-Scenario Breakdown

Set `SHOW_LABEL` to the `learner_label` of the run you want to inspect.

In [ ]:
SHOW_LABEL = summaries[-1]["learner_label"] if summaries else ""

SCENARIOS = [
    "skill_mastered__prereqs_met",
    "skill_mastered__prereqs_not_met",
    "skill_unmastered__prereqs_met",
    "skill_unmastered__prereqs_not_met",
]

target = next((s for s in summaries if s.get("learner_label") == SHOW_LABEL), None)
if target is None:
    print(f"Label not found: {SHOW_LABEL!r}")
else:
    agg = target["aggregate_scores"]
    conv_per_scen = agg["conversational"]["per_scenario"]
    lb_per_scen = agg["learning_behavior"]["per_scenario"]

    rows = []
    for scen in SCENARIOS:
        c = conv_per_scen.get(scen, {})
        lb = lb_per_scen.get(scen, {})
        rows.append(
            {
                "scenario": scen,
                "weight": c.get("weight"),
                "conv_macro_avg": c.get("macro_average_distance"),
                "q_w1": c.get("questions_per_interrogative_turn_w1"),
                "w_w1_log": c.get("avg_words_per_learner_turn_w1_log"),
                "error_jsd": c.get("error_types_jsd"),
                "talk_jsd": c.get("talk_moves_jsd"),
                "lb_macro_mae": lb.get("macro_mae"),
            }
        )

    display(
        pd.DataFrame(rows).style.format(
            "{:.4f}",
            subset=["weight", "conv_macro_avg", "q_w1", "w_w1_log", "error_jsd", "talk_jsd", "lb_macro_mae"],
            na_rep="—",
        )
    )

## 6 · Load Detail Records (JSONL)

In [ ]:
def load_details(summary: dict) -> list[dict]:
    details_path = Path(summary.get("details_file", ""))
    if not details_path.exists():
        details_path = Path(summary["_details_dir"]) / "dataset_fitted_conversational_results.jsonl"
    records = []
    with details_path.open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


# Change RUN_INDEX to inspect a different run
RUN_INDEX = 0
details = load_details(summaries[RUN_INDEX])
print(f"Loaded {len(details)} detail records from: {summaries[RUN_INDEX]['learner_label']}")

details_df = pd.DataFrame(
    [
        {
            "session_id": r.get("session_id"),
            "target_skill_id": r.get("target_skill_id"),
            "scenario": r.get("scenario"),
            "mastery_group": r.get("mastery_group"),
            "run_id": r.get("run_id"),
            "solution_found": r.get("solution_found"),
            "num_turns": r.get("num_turns"),
            "init_failed": r.get("initialization_failed", False),
            "has_any_talk_move": r.get("conversation_metrics", {}).get("has_any_talk_move"),
        }
        for r in details
    ]
)
display(details_df.head(10))

In [ ]:
# Compare few shot tutor vs. 0-shot tutor responses length:
# consider that 'summaries' was already filtered to keep relevant runs that can be compared

few_shot_response_length_totals, few_shot_convs_number = [0, 0], [0, 0]
for summary in summaries:
    det = load_details(summary)
    for r in det:
        few_shot_idx = int(r.get("few_shot_tutor_conversations_used", 0) > 0)
        for turn in r.get("dialogue_history", [])[1:]:  # skip initial assistant item prompt
            if turn.get("role") == "assistant":
                response = turn.get("content", "")
                few_shot_response_length_totals[few_shot_idx] += len(response.split())
                few_shot_convs_number[few_shot_idx] += 1

avg_response_lengths = [
    (total / few_shot_convs_number[idx]) if few_shot_convs_number[idx] > 0 else 0
    for idx, total in enumerate(few_shot_response_length_totals)
]
print(
    f"Average assistant response length with few-shot tutor: {avg_response_lengths[1]:.2f} words - across {few_shot_convs_number[1]} responses"
)
print(
    f"Average assistant response length with zero-shot tutor: {avg_response_lengths[0]:.2f} words - across {few_shot_convs_number[0]} responses"
)

## 7 · Conversation Browser

Set the filters below to display a specific conversation.

In [ ]:
# ---- Filters (set to None to skip) ----
FILTER_SKILL = None  # e.g. "MA.6.NSO.1.2"
FILTER_SCENARIO = None  # e.g. "skill_unmastered__prereqs_met"
FILTER_MASTERY = None  # e.g. "mastered" or "unmastered"
CONVERSATION_INDEX = 0  # which result to show within the filtered set
# ----------------------------------------


def render_conversation(record: dict) -> None:
    metrics = record.get("conversation_metrics", {})
    print(f"Session       : {record.get('session_id')}")
    print(f"Skill         : {record.get('target_skill_id')}")
    print(f"Scenario      : {record.get('scenario')}")
    print(f"Mastery group : {record.get('mastery_group')}")
    print(f"Run           : {record.get('run_id')}")
    print(f"Turns         : {record.get('num_turns')}")
    print(f"Solution found: {record.get('solution_found')}")
    print(f"Few shot tutor: {record.get('few_shot_tutor_conversations_used', 'N/A')} conversations used")
    if metrics:
        print(f"Talk moves    : {metrics.get('talk_moves')}")
        print(f"Errors        : {metrics.get('errors')}")
    print()
    print(f"Practice item : {record.get('practice_item_text', '')[:300]}")
    print("=" * 70)
    for turn in record.get("dialogue_history", []):
        role = turn.get("role", "?")
        label = "[Tutor  ]" if role == "assistant" else "[Student]"
        print(f"{label} {turn.get('content', '')}")
        print()


filtered = [
    r
    for r in details
    if not r.get("initialization_failed", False)
    and (FILTER_SKILL is None or r.get("target_skill_id") == FILTER_SKILL)
    and (FILTER_SCENARIO is None or r.get("scenario") == FILTER_SCENARIO)
    and (FILTER_MASTERY is None or r.get("mastery_group") == FILTER_MASTERY)
]

if not filtered:
    print("No records match the current filters.")
else:
    idx = CONVERSATION_INDEX % len(filtered)
    print(f"Showing {idx + 1} of {len(filtered)} matching records\n")
    render_conversation(filtered[idx])

In [ ]:
# Look at specific tutor responses metrics like average response length:

response_lengths = []
for record in details:
    if record.get("initialization_failed", False):
        continue
    convo = record.get("dialogue_history", [])
    for turn in convo:
        if turn.get("role") == "assistant":
            content = turn.get("content", "")
            response_lengths.append(len(content.split()))

## 8 · Random Sample for Human Validation

Pulls a random set of generated conversations with their LLM-as-judge classifications
to manually validate the talk-move and error-type labels.

Set `SAMPLE_SCENARIOS` to restrict sampling to specific scenarios.

In [ ]:
# Will sample from all the conversations from the filtered set for manual analysis

details = [load_details(summary) for summary in summaries]
for det in details:
    print(f"Loaded {len(det)} records from details")

In [ ]:
# ---- Sampling parameters ----
N_SAMPLE = 50  # number of conversations to sample
SAMPLE_SCENARIOS = None  # e.g. ["skill_unmastered__prereqs_met"] or None for all
RANDOM_SEED = 42
# -----------------------------

rng = random.Random(RANDOM_SEED)

pool = [
    r
    for detail in details
    for r in detail
    if not r.get("initialization_failed", False)
    and isinstance(r.get("conversation_metrics"), dict)
    and (SAMPLE_SCENARIOS is None or r.get("scenario") in SAMPLE_SCENARIOS)
]

sample = rng.sample(pool, min(N_SAMPLE, len(pool)))
print(f"Sampled {len(sample)} / {len(pool)} available conversations")
print(f"Scenarios: {sorted({r.get('scenario') for r in sample})}\n")

for i, record in enumerate(sample, start=1):
    print(f"\n{'=' * 70}")
    print(f"  Sample {i}/{len(sample)}")
    print(f"{'=' * 70}")
    render_conversation(record)

## 9 · Export Sample to CSV (for offline grading)

Produces a CSV with `human_talk_move_ok`, `human_error_type_ok`, and `human_notes` columns
for a grader to fill in.

In [ ]:
EXPORT_PATH = EVAL_OUTPUT_DIR / f"human_review/human_grading_sample_{len(sample)}.csv"
EXPORT_PATH.parent.mkdir(exist_ok=True, parents=True)

export_rows = []
for record in sample:
    metrics = record.get("conversation_metrics", {})
    conversation_text = "\n".join(f"[{t['role']}] {t['content']}" for t in record.get("dialogue_history", []))
    export_rows.append(
        {
            "session_id": record.get("session_id"),
            "target_skill_id": record.get("target_skill_id"),
            "scenario": record.get("scenario"),
            "mastery_group": record.get("mastery_group"),
            "run_id": record.get("run_id"),
            "solution_found": record.get("solution_found"),
            "num_turns": record.get("num_turns"),
            # LLM-as-judge labels
            "llm_talk_moves": json.dumps(metrics.get("talk_moves", {})),
            "llm_has_any_talk_move": metrics.get("has_any_talk_move"),
            "llm_errors": json.dumps(metrics.get("errors", {})),
            # Columns for the human grader to fill in
            "human_talk_move_ok": "",
            "human_error_type_ok": "",
            "human_notes": "",
            "conversation_text": conversation_text,
        }
    )

export_df = pd.DataFrame(export_rows)
export_df.to_csv(EXPORT_PATH, index=False)
print(f"Exported {len(export_df)} rows to {EXPORT_PATH.resolve()}")
display(
    export_df[
        ["session_id", "scenario", "llm_talk_moves", "llm_errors", "human_talk_move_ok", "human_error_type_ok"]
    ].head()
)

## 10 · Real vs. Simulated Distribution Plots

In [ ]:
try:
    import matplotlib.pyplot as plt

    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("matplotlib not installed — skipping plots. Install with: pip install matplotlib")

if HAS_MATPLOTLIB and summaries:
    target_summary = summaries[RUN_INDEX]
    agg = target_summary["aggregate_scores"]
    conv_per_scen = agg["conversational"]["per_scenario"]

    scen_keys = [s for s in SCENARIOS if s in conv_per_scen]
    n_scen = len(scen_keys)
    fig, axes = plt.subplots(2, n_scen, figsize=(5 * n_scen, 8))
    if n_scen == 1:
        axes = axes.reshape(2, 1)

    error_keys = ["NC", "CU", "PC", "SD", "SO"]
    talk_keys = ["asking_for_more_information", "making_a_claim", "providing_evidence_or_reasoning"]
    talk_short = ["ask_info", "claim", "evidence"]

    for col, scen in enumerate(scen_keys):
        detail = conv_per_scen[scen].get("detail", {})

        # Error types
        ax = axes[0, col]
        real_err = [detail.get("real_error_dist", {}).get(k, 0.0) for k in error_keys]
        sim_err = [detail.get("sim_error_dist", {}).get(k, 0.0) for k in error_keys]
        x = range(len(error_keys))
        ax.bar([v - 0.2 for v in x], real_err, width=0.4, label="Real", color="steelblue")
        ax.bar([v + 0.2 for v in x], sim_err, width=0.4, label="Simulated", color="salmon")
        ax.set_xticks(list(x))
        ax.set_xticklabels(error_keys)
        ax.set_title(f"Error types\n{scen}", fontsize=8)
        ax.set_ylim(0, 1)
        ax.legend(fontsize=8)

        # Talk moves
        ax = axes[1, col]
        real_talk = [detail.get("real_talk_dist", {}).get(k, 0.0) for k in talk_keys]
        sim_talk = [detail.get("sim_talk_dist", {}).get(k, 0.0) for k in talk_keys]
        xt = range(len(talk_keys))
        ax.bar([v - 0.2 for v in xt], real_talk, width=0.4, label="Real", color="steelblue")
        ax.bar([v + 0.2 for v in xt], sim_talk, width=0.4, label="Simulated", color="salmon")
        ax.set_xticks(list(xt))
        ax.set_xticklabels(talk_short, rotation=15)
        ax.set_title(f"Talk moves\n{scen}", fontsize=8)
        ax.set_ylim(0, 1)
        ax.legend(fontsize=8)

    plt.suptitle(
        f"Real vs. Simulated — {target_summary['learner_label']}",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()